# 03 — Compute Audit Table (`paper3_cross_model_audit_table.csv`)

Reads per-model M1–M5 summary CSVs from `INTERMEDIATE_DIR` (default `../results/`) and recomputes every column of the cross-model audit table.

| column | source file | rule |
|--------|-------------|------|
| `m1_layer` | `*_m1_gain_crossover.csv` | first layer `frac_above_1 >= M1_THRESH` |
| `m2_layer` | `*_m2_logit_lens_summary.csv` | first layer `top1_acc >= M2_THRESH` |
| `m3_layer` | `*_m3_similarity_summary.csv` | first layer `cos_final_mean >= M3_THRESH` |
| `m4_layer` | `*_m4_ablation_summary.csv` | `argmin(abl_kl_mean)` middle band (excl first/last 2) |
| `m5_layer` | `*_m5_alignment_summary.csv` | first layer `update_norm_mean >= profile mean` |
| `hourglass_edge_mid_ratio` | m4 ablation | edge-band mean KL / mid-band mean KL |
| `late_ramp_ratio` | m5 alignment | late-band `update_norm_mean` / shoulder mean |
| raw/cast stats | m2 logit-lens + casted | top-1 accuracy and knee layers |
| `m6_spearman_rho/p` | m6 per-prompt convergence | Spearman(`final_entropy`, `convergence_layer`) |

**Fallback**: when M1-M5 files are absent for a model, the row is read from the existing
`paper3_cross_model_audit_table.csv` and tagged `[archived]` in `data_source`.

Output: `RESULTS_DIR/paper3_cross_model_audit_table.csv`


In [ ]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from scipy import stats
warnings.filterwarnings("ignore", category=FutureWarning)

NOTEBOOK_DIR  = Path(os.path.abspath(""))
REPRODUCE_DIR = NOTEBOOK_DIR.parent
RESULTS_DIR   = REPRODUCE_DIR / "results"
INTERMEDIATE_DIR = Path(os.environ.get("INTERMEDIATE_DIR", str(RESULTS_DIR)))
EXISTING_AUDIT = RESULTS_DIR / "paper3_cross_model_audit_table.csv"
OUTPUT_AUDIT   = RESULTS_DIR / "paper3_cross_model_audit_table.csv"

cfg_path = REPRODUCE_DIR / "config.yaml"
with open(cfg_path) as fh:
    CFG = yaml.safe_load(fh)

M1_THRESH = CFG["experiment"]["m1_crossover_threshold"]
M2_THRESH = CFG["experiment"]["m2_crossover_threshold"]
M3_THRESH = CFG["experiment"]["m3_crossover_threshold"]

# Model registry: key = audit-table model name, prefix = file prefix used by nb01/02
# NOTE: Qwen audit key uses hyphens ("qwen2.5-1.5b") but file prefix uses underscores
# ("qwen2_1_5b") to match what notebooks 01 and 02 write.
MODEL_REGISTRY = {
    "gpt2": {
        "n_layers": CFG["models"]["gpt2"]["n_layers"],
        "prefix":   "gpt2",
        "hf_id":    CFG["models"]["gpt2"]["hf_id"],
    },
    "gemma2_2b": {
        "n_layers": CFG["models"]["gemma2_2b"]["n_layers"],
        "prefix":   "gemma2_2b",
        "hf_id":    CFG["models"]["gemma2_2b"]["hf_id"],
    },
    "qwen2.5-1.5b": {
        "n_layers": CFG["models"]["qwen2_1_5b"]["n_layers"],
        "prefix":   "qwen2_1_5b",   # underscore — matches nb01/02 output filenames
        "hf_id":    CFG["models"]["qwen2_1_5b"]["hf_id"],
    },
}

print("RESULTS_DIR      :", RESULTS_DIR)
print("INTERMEDIATE_DIR :", INTERMEDIATE_DIR)
print("Models           :", list(MODEL_REGISTRY.keys()))
print(f"Thresholds  M1={M1_THRESH}  M2={M2_THRESH}  M3={M3_THRESH}")


## Load per-model M1–M5 summary CSVs

For each model the notebook globs `INTERMEDIATE_DIR` for CSV files containing both the model prefix and the metric tag (case-insensitive). When multiple versioned copies exist the most recently modified is used. When no file is found the metric is filled from the archived audit table.


In [ ]:
def _find_csv(prefix, tag, search_dir):
    """Find a CSV in search_dir whose name contains both prefix and tag.

    Preference order (deterministic):
      1. Exact canonical name: {prefix}{tag}.csv  (no versioning suffix)
      2. First alphabetically among all matches (avoids mtime non-determinism)

    This ensures the reproduce package always picks the clean, unversioned file
    when both a canonical and a versioned copy (e.g. 'name(2).csv') exist.
    """
    candidates = sorted(Path(search_dir).glob("*.csv"))   # alphabetic, deterministic
    pl, tl = prefix.lower(), tag.lower()
    hits = [p for p in candidates if pl in p.name.lower() and tl in p.name.lower()]
    if not hits:
        return None
    # Prefer exact canonical: no parenthesised version suffix
    exact = [p for p in hits if "(" not in p.name]
    return exact[0] if exact else hits[0]


def load_metric_csv(model_key, metric_tag, search_dir):
    """Load a per-model metric CSV. Returns None if not found."""
    prefix = MODEL_REGISTRY[model_key]["prefix"]
    path   = _find_csv(prefix, metric_tag, search_dir)
    return None if path is None else pd.read_csv(path)


if EXISTING_AUDIT.exists():
    existing_audit = pd.read_csv(EXISTING_AUDIT)
    existing_audit["model"] = existing_audit["model"].astype(str)
    print("Loaded existing audit table:", EXISTING_AUDIT)
    display(existing_audit[["model","n_layers","m1_layer","m2_layer","m3_layer","m4_layer","m5_layer"]])
else:
    existing_audit = pd.DataFrame()
    print("WARNING: existing audit table not found - no fallback.")


def archived_value(model_key, col):
    """Return a value from the archived audit table, or None."""
    if existing_audit.empty: return None
    row = existing_audit[existing_audit["model"] == model_key]
    if row.empty or col not in row.columns: return None
    val = row.iloc[0][col]
    return None if pd.isna(val) else val


## Compute crossover layers per model

| Metric | Column | Rule |
|--------|--------|------|
| **M1** | `frac_above_1` | first layer >= `M1_THRESH` (0.50) |
| **M2** | `top1_acc` | first layer >= `M2_THRESH` (0.50) |
| **M3** | `cos_final_mean` | first layer >= `M3_THRESH` (0.50) |
| **M4** | `abl_kl_mean` | argmin in middle band (excl. first/last 2 layers) -- hourglass nadir |
| **M5** | `update_norm_mean` | first layer >= whole-profile mean -- late-ramp onset |

M4 and M5 are structural features, not simple threshold crossings.


In [ ]:
# ---- Crossover helpers ----
def find_crossover_layer(series, threshold, layers=None):
    """Return first layer where series >= threshold. Returns None if never reached."""
    mask = series >= threshold
    if not mask.any(): return None
    pos = int(mask.idxmax())
    if layers is not None:
        return int(layers.iloc[pos]) if hasattr(layers,"iloc") else int(layers[pos])
    return pos

def compute_m4_layer(df_m4):
    """Hourglass nadir: argmin(abl_kl_mean) in middle band (excl. first/last 2 rows)."""
    if df_m4 is None or len(df_m4) < 6: return None
    df_s = df_m4.sort_values("layer").reset_index(drop=True)
    nadir = df_s.iloc[2:-2]["abl_kl_mean"].idxmin()
    return int(df_s.loc[nadir, "layer"])

def compute_m5_layer(df_m5):
    """Late-ramp onset: first layer where update_norm_mean >= whole-profile mean."""
    if df_m5 is None or df_m5.empty: return None
    df_s = df_m5.sort_values("layer").reset_index(drop=True)
    mask = df_s["update_norm_mean"] >= df_s["update_norm_mean"].mean()
    if not mask.any(): return None
    return int(df_s.loc[int(mask.idxmax()), "layer"])

# ---- Ratio helpers ----
def compute_hourglass_ratio(df_m4, n_layers, early_n=3, late_n=3):
    """Ratio of edge-band mean abl_kl_mean to mid-band mean."""
    if df_m4 is None or df_m4.empty: return None
    df_s = df_m4.sort_values("layer").reset_index(drop=True)
    layers, kl = df_s["layer"].values, df_s["abl_kl_mean"].values
    edge = (layers <= layers[early_n-1]) | (layers >= layers[-late_n])
    mid  = ~edge
    em, mm = kl[edge].mean() if edge.any() else np.nan, kl[mid].mean() if mid.any() else np.nan
    if np.isnan(mm) or mm == 0: return None
    return float(em / mm)

def compute_late_ramp_ratio(df_m5, n_layers, late_n=4, mid_n=4):
    """Ratio of last late_n layers update_norm_mean to preceding mid_n layers."""
    if df_m5 is None or df_m5.empty: return None
    df_s = df_m5.sort_values("layer").reset_index(drop=True)
    if len(df_s) < late_n + mid_n: return None
    lm = df_s["update_norm_mean"].iloc[-late_n:].mean()
    mm = df_s["update_norm_mean"].iloc[-(late_n+mid_n):-late_n].mean()
    return None if mm == 0 else float(lm / mm)


In [ ]:
# ---- Raw/cast top-1 stats ----
def compute_raw_cast_stats(df_m2_full, df_m2_summary, df_cast_summary, early_end, late_start):
    """Derive top-1 accuracy stats and knee layers. All DFs optional (pass None if absent)."""
    out = {}
    if df_m2_full is not None and not df_m2_full.empty:
        df_s = df_m2_full.sort_values("layer")
        early_df = df_s[df_s["layer"] <= early_end]
        late_df  = df_s[df_s["layer"] >= late_start]
        out["raw_early_top1"] = float(early_df["top1_match"].mean()) if not early_df.empty else np.nan
        out["raw_late_top1"]  = float(late_df["top1_match"].mean())  if not late_df.empty  else np.nan
    elif df_m2_summary is not None and not df_m2_summary.empty:
        df_s = df_m2_summary.sort_values("layer")
        early_df = df_s[df_s["layer"] <= early_end]
        late_df  = df_s[df_s["layer"] >= late_start]
        out["raw_early_top1"] = float(early_df["top1_acc"].mean()) if not early_df.empty else np.nan
        out["raw_late_top1"]  = float(late_df["top1_acc"].mean())  if not late_df.empty  else np.nan
    else:
        out["raw_early_top1"] = out["raw_late_top1"] = np.nan
    if df_m2_summary is not None and not df_m2_summary.empty:
        df_s = df_m2_summary.sort_values("layer").reset_index(drop=True)
        out["raw_top1_50_layer"]       = find_crossover_layer(df_s["top1_acc"], 0.50, df_s["layer"])
        out["raw_top1_90_layer"]       = find_crossover_layer(df_s["top1_acc"], 0.90, df_s["layer"])
        out["raw_knee_from_cast_file"] = out["raw_top1_50_layer"]
    else:
        out["raw_top1_50_layer"] = out["raw_top1_90_layer"] = out["raw_knee_from_cast_file"] = np.nan
    if df_cast_summary is not None and not df_cast_summary.empty:
        df_s = df_cast_summary.sort_values("layer").reset_index(drop=True)
        early_df = df_s[df_s["layer"] <= early_end]
        out["cast_early_top1"]           = float(early_df["top1_acc"].mean()) if not early_df.empty else np.nan
        out["cast_knee"]                 = find_crossover_layer(df_s["top1_acc"], 0.50, df_s["layer"])
        out["raw_knee_from_cast_file"]   = find_crossover_layer(df_s["top1_acc"], M2_THRESH, df_s["layer"])
        out["cast_minus_raw_early_top1"] = out["cast_early_top1"] - out.get("raw_early_top1", np.nan)
    else:
        out["cast_early_top1"] = out["cast_knee"] = out["cast_minus_raw_early_top1"] = np.nan
    rk, ck = out.get("raw_knee_from_cast_file"), out.get("cast_knee")
    try:
        out["knee_shift_cast_minus_raw"] = int(ck) - int(rk)
    except (TypeError, ValueError):
        out["knee_shift_cast_minus_raw"] = np.nan
    return out

# ---- M6 Spearman stats ----
def compute_m6_stats(df_m6):
    """Spearman(final_entropy, convergence_layer) + per-difficulty means."""
    out = {}
    if df_m6 is None or df_m6.empty: return out
    conv_col    = "convergence_layer" if "convergence_layer" in df_m6.columns else None
    entropy_col = "final_entropy"     if "final_entropy"     in df_m6.columns else None
    if conv_col and entropy_col:
        valid = df_m6[[conv_col, entropy_col]].dropna()
        if len(valid) >= 3:
            rho, p = stats.spearmanr(valid[entropy_col], valid[conv_col])
            out["m6_spearman_rho"], out["m6_spearman_p"] = float(rho), float(p)
    if conv_col and "difficulty" in df_m6.columns:
        for diff in ("easy","medium","hard"):
            sub = df_m6[df_m6["difficulty"] == diff][conv_col].dropna()
            out[f"m6_{diff}_mean_layer"] = float(sub.mean()) if not sub.empty else np.nan
    return out


In [ ]:
# ==== Main per-model computation loop ====
records = []
for model_key, meta in MODEL_REGISTRY.items():
    n_layers   = meta["n_layers"]
    prefix     = meta["prefix"]      # file prefix (underscore form, e.g. qwen2_1_5b)
    early_end  = CFG["experiment"]["early_band_size"] - 1
    late_n_cfg = CFG["experiment"]["late_band_size"]
    late_start = n_layers - late_n_cfg
    sep = "=" * 60
    print("\n" + sep)
    print(f"Model: {model_key}  (n_layers={n_layers})")
    print(f"  searching: {INTERMEDIATE_DIR}")
    df_m1   = load_metric_csv(model_key, "_m1_gain_crossover",         INTERMEDIATE_DIR)
    df_m2s  = load_metric_csv(model_key, "_m2_logit_lens_summary",     INTERMEDIATE_DIR)
    df_m2f  = load_metric_csv(model_key, "_m2_logit_lens",             INTERMEDIATE_DIR)
    if df_m2f is not None and "prompt_idx" not in df_m2f.columns: df_m2f = None
    df_m3   = load_metric_csv(model_key, "_m3_similarity_summary",     INTERMEDIATE_DIR)
    df_m4   = load_metric_csv(model_key, "_m4_ablation_summary",       INTERMEDIATE_DIR)
    df_m5   = load_metric_csv(model_key, "_m5_alignment_summary",      INTERMEDIATE_DIR)
    df_cast = load_metric_csv(model_key, "_casted_logit_lens_summary", INTERMEDIATE_DIR)
    df_m6   = load_metric_csv(model_key, "_m6_per_prompt",             INTERMEDIATE_DIR)
    if df_m6 is None:
        df_m6 = load_metric_csv(model_key, "convergence_crossovers",   INTERMEDIATE_DIR)
    have = {"m1":df_m1 is not None,"m2":df_m2s is not None,
            "m3":df_m3 is not None,"m4":df_m4 is not None,"m5":df_m5 is not None}
    print("  Files found:", have)
    data_source = "computed" if all(have.values()) else "[archived]"

    # M1
    if df_m1 is not None:
        agg = df_m1.sort_values("layer").groupby("layer",as_index=False)["frac_above_1"].mean()
        m1_layer = find_crossover_layer(agg["frac_above_1"], M1_THRESH, agg["layer"])
    else:
        m1_layer = archived_value(model_key, "m1_layer")
        print(f"  m1_layer  <- archived: {m1_layer}")
    # M2
    if df_m2s is not None:
        s = df_m2s.sort_values("layer").reset_index(drop=True)
        m2_layer = find_crossover_layer(s["top1_acc"], M2_THRESH, s["layer"])
    else:
        m2_layer = archived_value(model_key, "m2_layer")
        print(f"  m2_layer  <- archived: {m2_layer}")
    # M3
    if df_m3 is not None:
        s = df_m3.sort_values("layer").reset_index(drop=True)
        m3_layer = find_crossover_layer(s["cos_final_mean"], M3_THRESH, s["layer"])
    else:
        m3_layer = archived_value(model_key, "m3_layer")
        print(f"  m3_layer  <- archived: {m3_layer}")
    # M4 (hourglass nadir)
    m4_layer = compute_m4_layer(df_m4)
    if m4_layer is None:
        m4_layer = archived_value(model_key, "m4_layer")
        print(f"  m4_layer  <- archived: {m4_layer}")
    # M5 (late-ramp onset)
    m5_layer = compute_m5_layer(df_m5)
    if m5_layer is None:
        m5_layer = archived_value(model_key, "m5_layer")
        print(f"  m5_layer  <- archived: {m5_layer}")
    # Boundary mean (M1,M2,M3,M5; M4 excluded by design)
    m_vals = [float(v) for v in (m1_layer,m2_layer,m3_layer,m5_layer)
              if v is not None and not (isinstance(v,float) and np.isnan(v))]
    boundary_mean_layer = float(np.mean(m_vals)) if m_vals else np.nan
    boundary_mean_depth = (
        boundary_mean_layer/n_layers if not np.isnan(boundary_mean_layer) else np.nan)
    # Hourglass + late-ramp
    hourglass_ratio = compute_hourglass_ratio(df_m4, n_layers)
    if hourglass_ratio is None:
        hourglass_ratio = archived_value(model_key, "hourglass_edge_mid_ratio")
        print(f"  hourglass <- archived: {hourglass_ratio}")
    late_ramp_ratio = compute_late_ramp_ratio(df_m5, n_layers, late_n=late_n_cfg)
    if late_ramp_ratio is None:
        late_ramp_ratio = archived_value(model_key, "late_ramp_ratio")
        print(f"  late_ramp <- archived: {late_ramp_ratio}")
    # Raw/cast with archive patch
    rc = compute_raw_cast_stats(df_m2f, df_m2s, df_cast, early_end, late_start)
    for k in ("raw_top1_50_layer","raw_top1_90_layer","raw_early_top1",
              "raw_late_top1","cast_early_top1","cast_minus_raw_early_top1",
              "raw_knee_from_cast_file","cast_knee","knee_shift_cast_minus_raw"):
        v = rc.get(k)
        if v is None or (isinstance(v,float) and np.isnan(v)):
            av = archived_value(model_key, k)
            if av is not None: rc[k] = av
    # M6
    m6 = compute_m6_stats(df_m6)
    if not m6:
        for k in ("m6_spearman_rho","m6_spearman_p",
                  "m6_easy_mean_layer","m6_medium_mean_layer","m6_hard_mean_layer"):
            av = archived_value(model_key, k)
            if av is not None: m6[k] = av
        if m6: print("  m6 stats  <- archived")
    def _src(tag):
        p = _find_csv(prefix, tag, INTERMEDIATE_DIR)
        return p.name if p else "[archived]"
    src_crossovers = _src("_m1_gain_crossover")
    src_m2         = _src("_m2_logit_lens_summary")
    src_cast       = _src("_casted_logit_lens_summary")
    m6_path = _find_csv(prefix, "_m6_per_prompt", INTERMEDIATE_DIR)
    src_m6  = m6_path.name if m6_path else _src("convergence_crossovers")
    rec = {
        "model": model_key, "n_layers": n_layers,
        "m1_layer": m1_layer, "m2_layer": m2_layer, "m3_layer": m3_layer,
        "m4_layer": m4_layer, "m5_layer": m5_layer,
        "boundary_mean_layer_m1m2m3m5": boundary_mean_layer,
        "boundary_mean_depth_m1m2m3m5": boundary_mean_depth,
        "raw_top1_50_layer": rc.get("raw_top1_50_layer"),
        "raw_top1_90_layer": rc.get("raw_top1_90_layer"),
        "raw_early_top1": rc.get("raw_early_top1"),
        "raw_late_top1": rc.get("raw_late_top1"),
        "cast_early_top1": rc.get("cast_early_top1"),
        "cast_minus_raw_early_top1": rc.get("cast_minus_raw_early_top1"),
        "raw_knee_from_cast_file": rc.get("raw_knee_from_cast_file"),
        "cast_knee": rc.get("cast_knee"),
        "knee_shift_cast_minus_raw": rc.get("knee_shift_cast_minus_raw"),
        "hourglass_edge_mid_ratio": hourglass_ratio,
        "late_ramp_ratio": late_ramp_ratio,
        "m6_spearman_rho": m6.get("m6_spearman_rho"),
        "m6_spearman_p": m6.get("m6_spearman_p"),
        "m6_easy_mean_layer": m6.get("m6_easy_mean_layer"),
        "m6_medium_mean_layer": m6.get("m6_medium_mean_layer"),
        "m6_hard_mean_layer": m6.get("m6_hard_mean_layer"),
        "source_crossovers": src_crossovers, "source_m2": src_m2,
        "source_cast": src_cast, "source_m6": src_m6,
        "data_source": data_source,
    }
    records.append(rec)
    print(f"  => m1={m1_layer}  m2={m2_layer}  m3={m3_layer}  "
          f"m4={m4_layer}  m5={m5_layer}  source={data_source}")


## Assemble audit table

Combine all model records into a single DataFrame, display it, and write `paper3_cross_model_audit_table.csv` to `RESULTS_DIR`.


In [ ]:
df_audit = pd.DataFrame(records)
COL_ORDER = [
    "model", "n_layers",
    "m1_layer", "m2_layer", "m3_layer", "m4_layer", "m5_layer",
    "boundary_mean_layer_m1m2m3m5", "boundary_mean_depth_m1m2m3m5",
    "raw_top1_50_layer", "raw_top1_90_layer",
    "raw_early_top1", "raw_late_top1",
    "cast_early_top1", "cast_minus_raw_early_top1",
    "raw_knee_from_cast_file", "cast_knee", "knee_shift_cast_minus_raw",
    "hourglass_edge_mid_ratio", "late_ramp_ratio",
    "m6_spearman_rho", "m6_spearman_p",
    "m6_easy_mean_layer", "m6_medium_mean_layer", "m6_hard_mean_layer",
    "source_crossovers", "source_m2", "source_cast", "source_m6",
    "data_source",
]
extra_cols = [c for c in df_audit.columns if c not in COL_ORDER]
df_audit   = df_audit[COL_ORDER + extra_cols]
print("Audit table shape:", df_audit.shape)
display(df_audit)
OUTPUT_AUDIT.parent.mkdir(parents=True, exist_ok=True)
df_audit.to_csv(OUTPUT_AUDIT, index=False)
print("\nWritten: " + str(OUTPUT_AUDIT))
archived_models = df_audit[df_audit["data_source"] == "[archived]"]["model"].tolist()
computed_models = df_audit[df_audit["data_source"] == "computed"]["model"].tolist()
if computed_models: print(f"Computed from live CSVs : {computed_models}")
if archived_models: print(f"Archived (fallback)     : {archived_models}")


## Spot-check verification

Compare recomputed values against the known-good values from the prior audit table (validated 2026-02).

- **Archived rows** (`data_source = [archived]`): values must match **exactly** (they were read from the same file).
- **Computed rows** (`data_source = computed`): integer crossover layers allow **+/-1 layer** tolerance.
- `n_layers` must always match exactly.


In [ ]:
# Known-good ground truth (paper3_cross_model_audit_table.csv, validated 2026-02)
GROUND_TRUTH = {
    "gpt2":         {"n_layers":12,"m1_layer":9,"m2_layer":9,"m3_layer":10,"m4_layer":4,"m5_layer":8},
    "gemma2_2b":    {"n_layers":26,"m1_layer":13,"m2_layer":5,"m3_layer":24,"m4_layer":8,"m5_layer":13},
    "qwen2.5-1.5b": {"n_layers":28,"m1_layer":21,"m2_layer":24,"m3_layer":27,"m4_layer":9,"m5_layer":23},
}
LAYER_TOL  = 1
EXACT_COLS = {"n_layers"}
df_check = pd.read_csv(OUTPUT_AUDIT)
results, all_pass = [], True
for model_key, truth in GROUND_TRUTH.items():
    row = df_check[df_check["model"] == model_key]
    if row.empty:
        results.append({"model":model_key,"col":"(row)","status":"FAIL",
                        "actual":"MISSING","expected":"","note":""})
        all_pass = False; continue
    row = row.iloc[0]
    is_archived = str(row.get("data_source","")).strip() == "[archived]"
    for col, expected in truth.items():
        actual = row.get(col, None)
        if actual is None or (isinstance(actual,float) and np.isnan(actual)):
            status, note, disp = "FAIL", "value missing", "NaN"
            all_pass = False
        elif col in EXACT_COLS:
            ok = int(actual) == int(expected)
            status = "PASS" if ok else "FAIL"
            note, disp = "exact", str(int(actual))
            if not ok: all_pass = False
        else:
            tol  = 0 if is_archived else LAYER_TOL
            ok   = abs(int(actual) - int(expected)) <= tol
            status = "PASS" if ok else "FAIL"
            note  = "archived" if is_archived else f"computed tol={tol}"
            disp  = str(int(actual))
            if not ok: all_pass = False
        results.append({"model":model_key,"col":col,"status":status,
                        "actual":disp,"expected":expected,"note":note})
df_results = pd.DataFrame(results)
display(df_results)
print("\n" + "=" * 60)
if all_pass:
    print("ALL CHECKS PASSED")
else:
    n_fail = (df_results["status"] == "FAIL").sum()
    print(f"FAILED: {n_fail} check(s) - review table above.")
print(f"\nOutput  : {OUTPUT_AUDIT}")
print(f"Rows    : {len(df_check)}")
print(f"Columns : {len(df_check.columns)}")
print("\nColumn list:")
for c in df_check.columns: print(f"  {c}")
